# Notebook 11 Part 1: Building an ETL Pipeline in PySpark


### Pipeline
```text
Source → Ingest → Validate → Clean → Transform → Load → Verify
```

## 1. What Is ETL?

```text
E = Extract
T = Transform
L = Load
```

**Extract** means reading data from sources such as CSV, JSON, Parquet, databases or APIs.

**Transform** means cleaning, validating and applying business logic.

**Load** means writing the final data to a destination such as Parquet, Delta Lake, a database or a data warehouse.

## 2. Pipeline Used in This Notebook

```text
CSV Source
   ↓
Read Data
   ↓
Validate Schema
   ↓
Clean Nulls
   ↓
Remove Duplicates
   ↓
Convert Date
   ↓
Apply Business Rules
   ↓
Create Derived Columns
   ↓
Write Parquet
   ↓
Read Back and Verify
```

## 3. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["PATH"] = os.environ["HADOOP_HOME"] + "/bin;" + os.environ["PATH"]

## 4. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("PySpark_ETL_Pipeline") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## 5. Create Demo Source File

In [3]:
from pathlib import Path

base_path = Path("pyspark_etl_demo")
base_path.mkdir(parents=True, exist_ok=True)

source_file = base_path / "sales_source.csv"

source_file.write_text(
    '''order_id,customer_name,city,product,amount,order_date
101,Amit,Delhi,Laptop,55000,2026-09-01
102,Priya,Mumbai,Mobile,30000,2026-09-01
103,Ravi,Delhi,Tablet,25000,2026-09-02
104,Neha,Pune,Laptop,60000,2026-09-02
104,Neha,Pune,Laptop,60000,2026-09-02
105,Karan,,Mobile,35000,2026-09-03
106,Simran,Delhi,Monitor,,2026-09-03
107,Rahul,Mumbai,Laptop,70000,2026-09-04
108,Anjali,Pune,Tablet,28000,invalid_date
''',
    encoding="utf-8"
)

print(source_file.resolve())

E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_etl_demo\sales_source.csv


## 6. EXTRACT: Read CSV Source

In [4]:
raw_df = spark.read.csv(
    str(source_file),
    header=True,
    inferSchema=True
)

raw_df.show(truncate=False)

+--------+-------------+------+-------+------+------------+
|order_id|customer_name|city  |product|amount|order_date  |
+--------+-------------+------+-------+------+------------+
|101     |Amit         |Delhi |Laptop |55000 |2026-09-01  |
|102     |Priya        |Mumbai|Mobile |30000 |2026-09-01  |
|103     |Ravi         |Delhi |Tablet |25000 |2026-09-02  |
|104     |Neha         |Pune  |Laptop |60000 |2026-09-02  |
|104     |Neha         |Pune  |Laptop |60000 |2026-09-02  |
|105     |Karan        |NULL  |Mobile |35000 |2026-09-03  |
|106     |Simran       |Delhi |Monitor|NULL  |2026-09-03  |
|107     |Rahul        |Mumbai|Laptop |70000 |2026-09-04  |
|108     |Anjali       |Pune  |Tablet |28000 |invalid_date|
+--------+-------------+------+-------+------+------------+



## 7. Inspect Schema

In [5]:
raw_df.printSchema()

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = true)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = true)
 |-- order_date: string (nullable = true)



## 8. Validate Raw Record Count

In [6]:
raw_count = raw_df.count()
print("Raw record count:", raw_count)

Raw record count: 9


## 9. Check Null Values

In [12]:
from pyspark.sql.functions import col, sum as spark_sum

raw_df.select(
    [
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in raw_df.columns
    ]
).show()

+--------+-------------+----+-------+------+----------+
|order_id|customer_name|city|product|amount|order_date|
+--------+-------------+----+-------+------+----------+
|       0|            0|   1|      0|     1|         0|
+--------+-------------+----+-------+------+----------+



## 10. Check Duplicate Records

In [13]:
duplicate_count = raw_df.count() - raw_df.dropDuplicates().count()
print("Duplicate records:", duplicate_count)

Duplicate records: 1


## 11. TRANSFORM — Remove Duplicates

In [14]:
clean_df = raw_df.dropDuplicates()
print("Rows after duplicate removal:", clean_df.count())

Rows after duplicate removal: 8


## 12. Handle Missing City

In [15]:
clean_df = clean_df.fillna({"city": "Unknown"})
clean_df.show()

+--------+-------------+-------+-------+------+------------+
|order_id|customer_name|   city|product|amount|  order_date|
+--------+-------------+-------+-------+------+------------+
|     103|         Ravi|  Delhi| Tablet| 25000|  2026-09-02|
|     108|       Anjali|   Pune| Tablet| 28000|invalid_date|
|     104|         Neha|   Pune| Laptop| 60000|  2026-09-02|
|     102|        Priya| Mumbai| Mobile| 30000|  2026-09-01|
|     107|        Rahul| Mumbai| Laptop| 70000|  2026-09-04|
|     101|         Amit|  Delhi| Laptop| 55000|  2026-09-01|
|     105|        Karan|Unknown| Mobile| 35000|  2026-09-03|
|     106|       Simran|  Delhi|Monitor|  NULL|  2026-09-03|
+--------+-------------+-------+-------+------+------------+



## 13. Handle Missing Amount

For this beginner pipeline:

```text
missing amount → 0
```

In a real project, the business rule may instead reject or quarantine the row.

In [16]:
clean_df = clean_df.fillna({"amount": 0})
clean_df.show()

+--------+-------------+-------+-------+------+------------+
|order_id|customer_name|   city|product|amount|  order_date|
+--------+-------------+-------+-------+------+------------+
|     103|         Ravi|  Delhi| Tablet| 25000|  2026-09-02|
|     108|       Anjali|   Pune| Tablet| 28000|invalid_date|
|     104|         Neha|   Pune| Laptop| 60000|  2026-09-02|
|     102|        Priya| Mumbai| Mobile| 30000|  2026-09-01|
|     107|        Rahul| Mumbai| Laptop| 70000|  2026-09-04|
|     101|         Amit|  Delhi| Laptop| 55000|  2026-09-01|
|     105|        Karan|Unknown| Mobile| 35000|  2026-09-03|
|     106|       Simran|  Delhi|Monitor|     0|  2026-09-03|
+--------+-------------+-------+-------+------+------------+



## 14. Convert order_date to Date Type

In [17]:
from pyspark.sql.functions import try_to_date

clean_df = clean_df.withColumn(
    "order_date",
    try_to_date(
        col("order_date"),
        "yyyy-MM-dd"
    )
)
clean_df.show()
clean_df.printSchema()

+--------+-------------+-------+-------+------+----------+
|order_id|customer_name|   city|product|amount|order_date|
+--------+-------------+-------+-------+------+----------+
|     103|         Ravi|  Delhi| Tablet| 25000|2026-09-02|
|     108|       Anjali|   Pune| Tablet| 28000|      NULL|
|     104|         Neha|   Pune| Laptop| 60000|2026-09-02|
|     102|        Priya| Mumbai| Mobile| 30000|2026-09-01|
|     107|        Rahul| Mumbai| Laptop| 70000|2026-09-04|
|     101|         Amit|  Delhi| Laptop| 55000|2026-09-01|
|     105|        Karan|Unknown| Mobile| 35000|2026-09-03|
|     106|       Simran|  Delhi|Monitor|     0|2026-09-03|
+--------+-------------+-------+-------+------+----------+

root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- city: string (nullable = false)
 |-- product: string (nullable = true)
 |-- amount: integer (nullable = false)
 |-- order_date: date (nullable = true)



## 15. Identify Invalid Date Records

In [18]:
invalid_date_df = clean_df.filter(
    col("order_date").isNull()
)

invalid_date_df.show()

+--------+-------------+----+-------+------+----------+
|order_id|customer_name|city|product|amount|order_date|
+--------+-------------+----+-------+------+----------+
|     108|       Anjali|Pune| Tablet| 28000|      NULL|
+--------+-------------+----+-------+------+----------+



## 16. Remove Invalid Date Records

In [19]:
clean_df = clean_df.filter(
    col("order_date").isNotNull()
)

print("Rows after date validation:", clean_df.count())

Rows after date validation: 7


## 17. Keep Only Valid Sales Amounts

In [20]:
clean_df = clean_df.filter(
    col("amount") > 0
)

clean_df.show()

+--------+-------------+-------+-------+------+----------+
|order_id|customer_name|   city|product|amount|order_date|
+--------+-------------+-------+-------+------+----------+
|     103|         Ravi|  Delhi| Tablet| 25000|2026-09-02|
|     104|         Neha|   Pune| Laptop| 60000|2026-09-02|
|     102|        Priya| Mumbai| Mobile| 30000|2026-09-01|
|     107|        Rahul| Mumbai| Laptop| 70000|2026-09-04|
|     101|         Amit|  Delhi| Laptop| 55000|2026-09-01|
|     105|        Karan|Unknown| Mobile| 35000|2026-09-03|
+--------+-------------+-------+-------+------+----------+



## 18. Create Sales Category

Business rule:

```text
amount >= 50000 → High Value
amount >= 30000 → Medium Value
otherwise       → Low Value
```

In [21]:
from pyspark.sql.functions import when

transformed_df = clean_df.withColumn(
    "sales_category",
    when(col("amount") >= 50000, "High Value")
    .when(col("amount") >= 30000, "Medium Value")
    .otherwise("Low Value")
)

transformed_df.show()

+--------+-------------+-------+-------+------+----------+--------------+
|order_id|customer_name|   city|product|amount|order_date|sales_category|
+--------+-------------+-------+-------+------+----------+--------------+
|     103|         Ravi|  Delhi| Tablet| 25000|2026-09-02|     Low Value|
|     104|         Neha|   Pune| Laptop| 60000|2026-09-02|    High Value|
|     102|        Priya| Mumbai| Mobile| 30000|2026-09-01|  Medium Value|
|     107|        Rahul| Mumbai| Laptop| 70000|2026-09-04|    High Value|
|     101|         Amit|  Delhi| Laptop| 55000|2026-09-01|    High Value|
|     105|        Karan|Unknown| Mobile| 35000|2026-09-03|  Medium Value|
+--------+-------------+-------+-------+------+----------+--------------+



## 19. Create Tax Amount

In [22]:
transformed_df = transformed_df.withColumn(
    "tax_amount",
    col("amount") * 0.18
)

## 20. Create Final Amount

In [23]:
transformed_df = transformed_df.withColumn(
    "final_amount",
    col("amount") + col("tax_amount")
)

transformed_df.show()

+--------+-------------+-------+-------+------+----------+--------------+----------+------------+
|order_id|customer_name|   city|product|amount|order_date|sales_category|tax_amount|final_amount|
+--------+-------------+-------+-------+------+----------+--------------+----------+------------+
|     103|         Ravi|  Delhi| Tablet| 25000|2026-09-02|     Low Value|    4500.0|     29500.0|
|     104|         Neha|   Pune| Laptop| 60000|2026-09-02|    High Value|   10800.0|     70800.0|
|     102|        Priya| Mumbai| Mobile| 30000|2026-09-01|  Medium Value|    5400.0|     35400.0|
|     107|        Rahul| Mumbai| Laptop| 70000|2026-09-04|    High Value|   12600.0|     82600.0|
|     101|         Amit|  Delhi| Laptop| 55000|2026-09-01|    High Value|    9900.0|     64900.0|
|     105|        Karan|Unknown| Mobile| 35000|2026-09-03|  Medium Value|    6300.0|     41300.0|
+--------+-------------+-------+-------+------+----------+--------------+----------+------------+



## 21. Select Final Required Columns

In [24]:
final_df = transformed_df.select(
    "order_id",
    "customer_name",
    "city",
    "product",
    "amount",
    "tax_amount",
    "final_amount",
    "sales_category",
    "order_date"
)

final_df.show(truncate=False)

+--------+-------------+-------+-------+------+----------+------------+--------------+----------+
|order_id|customer_name|city   |product|amount|tax_amount|final_amount|sales_category|order_date|
+--------+-------------+-------+-------+------+----------+------------+--------------+----------+
|103     |Ravi         |Delhi  |Tablet |25000 |4500.0    |29500.0     |Low Value     |2026-09-02|
|104     |Neha         |Pune   |Laptop |60000 |10800.0   |70800.0     |High Value    |2026-09-02|
|102     |Priya        |Mumbai |Mobile |30000 |5400.0    |35400.0     |Medium Value  |2026-09-01|
|107     |Rahul        |Mumbai |Laptop |70000 |12600.0   |82600.0     |High Value    |2026-09-04|
|101     |Amit         |Delhi  |Laptop |55000 |9900.0    |64900.0     |High Value    |2026-09-01|
|105     |Karan        |Unknown|Mobile |35000 |6300.0    |41300.0     |Medium Value  |2026-09-03|
+--------+-------------+-------+-------+------+----------+------------+--------------+----------+



## 22. Validate Final Record Count

In [25]:
final_count = final_df.count()

print("Raw count:", raw_count)
print("Final count:", final_count)

Raw count: 9
Final count: 6


## 23. Check Final Null Values

In [26]:
final_df.select(
    [
        spark_sum(col(c).isNull().cast("int")).alias(c)
        for c in final_df.columns
    ]
).show()

+--------+-------------+----+-------+------+----------+------------+--------------+----------+
|order_id|customer_name|city|product|amount|tax_amount|final_amount|sales_category|order_date|
+--------+-------------+----+-------+------+----------+------------+--------------+----------+
|       0|            0|   0|      0|     0|         0|           0|             0|         0|
+--------+-------------+----+-------+------+----------+------------+--------------+----------+



## 24. LOAD — Define Output Path

In [27]:
output_path = base_path / "output" / "sales_cleaned_parquet"
print(output_path.resolve())

E:\data\Documents\Data_Engineering\Module4-Pyspark\pyspark_etl_demo\output\sales_cleaned_parquet


## 25. Write Final Data as Parquet

Parquet is commonly used in data engineering because it stores schema, uses columnar storage and supports compression.

In [28]:
print(Path("C:/hadoop/bin/winutils.exe").exists())
print(Path("C:/hadoop/bin/hadoop.dll").exists())

True
True


In [29]:
final_df.write \
    .mode("overwrite") \
    .parquet(str(output_path))

**Windows note:** if this write fails with a Hadoop native-library error, the issue is your Windows Hadoop setup, not the ETL logic.

## 26. Read Loaded Data Back

In [30]:
loaded_df = spark.read.parquet(
    str(output_path)
)

loaded_df.show()
loaded_df.printSchema()

+--------+-------------+-------+-------+------+----------+------------+--------------+----------+
|order_id|customer_name|   city|product|amount|tax_amount|final_amount|sales_category|order_date|
+--------+-------------+-------+-------+------+----------+------------+--------------+----------+
|     103|         Ravi|  Delhi| Tablet| 25000|    4500.0|     29500.0|     Low Value|2026-09-02|
|     104|         Neha|   Pune| Laptop| 60000|   10800.0|     70800.0|    High Value|2026-09-02|
|     102|        Priya| Mumbai| Mobile| 30000|    5400.0|     35400.0|  Medium Value|2026-09-01|
|     107|        Rahul| Mumbai| Laptop| 70000|   12600.0|     82600.0|    High Value|2026-09-04|
|     101|         Amit|  Delhi| Laptop| 55000|    9900.0|     64900.0|    High Value|2026-09-01|
|     105|        Karan|Unknown| Mobile| 35000|    6300.0|     41300.0|  Medium Value|2026-09-03|
+--------+-------------+-------+-------+------+----------+------------+--------------+----------+

root
 |-- order_id:

## 27. Validate Output Count

In [31]:
loaded_count = loaded_df.count()

print("Final DataFrame count:", final_count)
print("Loaded output count:", loaded_count)

Final DataFrame count: 6
Loaded output count: 6


## 28. Add Simple ETL Assertion

In [ ]:
assert final_count == loaded_count
print("ETL validation passed.")

## 29. ETL Summary

```text
EXTRACT
Read CSV

VALIDATE
Check schema
Count rows
Check nulls
Check duplicates

CLEAN
Remove duplicates
Fill missing city
Handle missing amount
Reject invalid dates
Reject invalid amount

TRANSFORM
Create sales_category
Create tax_amount
Create final_amount

LOAD
Write Parquet

VERIFY
Read output back
Compare counts
```

## 30. ETL Best Practices

1. Keep raw data separate from processed data.
2. Track row counts at important stages.
3. Validate schema before transformations.
4. Define clear rules for invalid records.
5. Write only required columns.
6. Prefer Parquet/Delta for processed analytical data.
7. Verify that output can be read successfully.

## 31. Real-World Layered Architecture

```text
Source Systems
      ↓
Raw / Bronze
      ↓
Validation + Cleaning
      ↓
Silver / Curated
      ↓
Business Aggregations
      ↓
Gold
      ↓
BI / ML / Reporting
```

We will connect this pattern with Delta Lake later in Phase 4.

## 34. Final Takeaway

```text
ETL = Extract + Transform + Load
```

A good ETL pipeline does not only transform data.

It also validates the source, applies clear data-quality rules, loads the result and verifies that the output is correct.

## 35. Stop SparkSession

In [46]:
spark.stop()